#### Imports

In [18]:
import numpy as np
import matplotlib.pyplot as plt
import struct
import math

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from tqdm import tqdm
import time
import random

#### Useful functions

In [48]:
### Takes a box and splits it into cubes (blocks) of a specified dimension. Returns
### a list of numpy arrays, each representing one block of the volume. If block_dim
### does not divide equally into the volume dimensions, creates blocks that extend past the
### dimensions of the volume, and populates the coordinates outside the volume with a chosen 
### junk, which is an input to the function
def to_blocks(box, block_dim, junk):
    
    xdim = box.shape[0]
    ydim = box.shape[1]
    zdim = box.shape[2]

    x_steps = int(xdim // block_dim)
    if xdim % block_dim != 0:
        x_steps += 1
    if xdim < block_dim:
        x_steps = 1
    y_steps = int(ydim / block_dim)
    if ydim % block_dim != 0:
        y_steps += 1
    if ydim < block_dim:
        y_steps = 1
    z_steps = int(zdim / block_dim)
    if zdim % block_dim != 0:
        z_steps += 1
    if zdim < block_dim:
        z_steps = 1
    num_blocks = int(x_steps * y_steps * z_steps)

    blocks = []

    for k in range(z_steps):
        for j in range(y_steps):
            for i in range(x_steps):

                x_lo = int(i * block_dim)
                y_lo = int(j * block_dim)
                z_lo = int(k * block_dim)

                block = np.empty((block_dim, block_dim, block_dim))

                for z in range(block_dim):
                    for y in range(block_dim):
                        for x in range(block_dim):

                            if (x_lo + x) >= xdim:
                                block[x][y][z] = junk
                            elif (y_lo + y) >= ydim:
                                block[x][y][z] = junk
                            elif (z_lo + z) >= zdim:
                                block[x][y][z] = junk
                            else:
                                block[x][y][z] = box[x_lo + x][y_lo + y][z_lo + z]
                
                blocks.append(block)

    return blocks



### Reads binary files in input_dir between timesteps min_file and max_file,
### and returns them in a list of numpy arrays, one for each chunk of data (box).
### Also returns a list of the tuples of the location and dimension of each
### box. Finally, returns a list of the number of boxes at each time step.
def process_data(input_dir, min_file, max_file, level, component):

    print("Retrieving data from directory", input_dir)
    # create output lists
    boxes = []
    locations = []
    dimensions = []
    box_counts = []

    print("Processing data...")
    for t in tqdm(range(min_file, max_file)): # iterate over each time step
        filename = input_dir + str(t) + "-wholeNewFormat-" + str(component) + "-" + str(level) + ".raw"

        # Keeps track of how many boxes are at each time step
        box_count = 0
        
        with open(filename, "rb") as file:
            while True: # Iterate through all the data at the current timestep until none is left
                test = file.read(4)
                if len(test) < 4: # Break the loop if all data has been read
                    break
                # Read the location of the box
                x = int(struct.unpack('<f', test)[0])
                y = int(struct.unpack('<f', file.read(4))[0])
                z = int(struct.unpack('<f', file.read(4))[0])
                locations.append((x, y, z))

                # Read the dimensions of the box
                xdim = int(struct.unpack('<f', file.read(4))[0])
                ydim = int(struct.unpack('<f', file.read(4))[0])
                zdim = int(struct.unpack('<f', file.read(4))[0])
                dimensions.append((xdim, ydim, zdim))

                # Read the data in the box
                box = np.empty((xdim, ydim, zdim))
                for k in range(zdim):
                    for j in range(ydim):
                        for i in range(xdim):
                            box[i][j][k] = struct.unpack('<f', file.read(4))[0]
                boxes.append(box)
                box_count += 1
                
        box_counts.append(box_count)

    return boxes, locations, dimensions, box_counts



def create_samples(boxes, block_dim, junk, noise):

    print("Creating samples...")
    
    # Check if block_dim is a power of 2 (necessary for convolutional autoencoder)
    if not math.log2(block_dim).is_integer():
        print("Invalid block_dim! Dimension must be a power of 2.")
        return
    
    samples = []
    for i in tqdm(range(len(boxes))): # Iterate through each box (now we don't care which time step
                                      # it's associated with
        box = boxes[i]
        blocks = to_blocks(box, block_dim, junk)
        for l in range(len(blocks)): # iterate through each block and add it to the list of inputs
            if noise: # add noise if desired
                for r in range(3): # iterate to create three instances, two with some noise
                    input = blocks[l]
                    if r == 0:
                        samples.append(input)
                    else:         
                        for z in range(block_dim):
                            for y in range(block_dim):
                                for x in range(block_dim):
                                    noise = 0.01 * ((-1)**r) # create some noise
                                    input[x,y,z] = input[x,y,z] + noise # add noise to the input
                        samples.append(input)

            else:
                input = blocks[l]
                samples.append(input)

    samples = np.array(samples)
    num_samples = samples.shape[0]
    samples = samples.reshape(num_samples, -1)
    return samples



def initialize_centroids(blocks, num_centroids):
    random_indices = random.sample(range(len(blocks)), num_centroids)
    centroids = []
    for i in random_indices:
        centroids.append(i)
    return centroids


def assign_to_centroid(blocks, centroids):
    assignments = []
    for block in blocks:
        distances = []
        for centroid in centroids:
            distance = np.linalg.norm(block - centroid)
            distances.append(distance)
        assignments.append(np.argmin(distances))
    return assignments


def update_centroids(blocks, assignments, num_centroids, old_centroids, block_dim):
    new_centroids = np.zeros((num_centroids, block_dim**3))
    counts = np.zeros(num_centroids)

    for i, block in enumerate(blocks):
        centroid_idx = assignments[i]
        new_centroids[centroid_idx] += block
        counts[centroid_idx] += 1

    # Average the blocks for each centroid
    for i in range(num_centroids):
        if counts[i] > 0:
            new_centroids[i] /= counts[i]
        else:
            new_centroids[i] = old_centroids[i]

    return new_centroids

#### Hyperparameters

In [54]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on", device)
min_file = 74
max_file = 75
block_dim = 4
level = 0
component = 6
junk = 0
max_iter = 100
tol = 1e-4
codebook_size = 256
dir = "wholeVolumesNewFormat-" + str(component) + "-" + str(level) + "/"

Running on cuda


#### Data preprocessing

In [55]:
boxes, locations, dimensions, num_boxes = process_data(dir, min_file, max_file, level, component)

blocks = create_samples(boxes, block_dim, junk, False)
print("Number of blocks:", blocks.shape[0])
print(blocks.shape)

Retrieving data from directory wholeVolumesNewFormat-6-0/
Processing data...


100%|█████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  1.07it/s]


Creating samples...


100%|████████████████████████████████████████████████████████████████████████████████| 576/576 [00:01<00:00, 436.40it/s]

Number of blocks: 36864
(36864, 64)


#### Generalized Llyod Algorithm to create codebook

In [56]:
centroids = initialize_centroids(blocks, codebook_size)
prev_centroids = centroids.copy()

for iteration in tqdm(range(max_iter)):
    assignments = assign_to_centroid(blocks, centroids)
    centroids = update_centroids(blocks, assignments, codebook_size, prev_centroids, block_dim)

    delta = np.sum([np.linalg.norm(centroids[i] - prev_centroids[i]) for i in range(codebook_size)])
    if delta < tol:
        print(f"Converged after {iteration + 1} iterations.")
        break

    prev_centroids = centroids.copy()

codebook = (assignments, centroids)

100%|█████████████████████████████████████████████████████████████████████████████████| 100/100 [36:45<00:00, 22.06s/it]


#### Reconsruct timestep from codebook